# 03 — Exploratory Data Analysis (EDA)

**Business Question:** What borrower, loan, and geographic factors drive loan default rates across 13 years of US peer-to-peer lending?

**Input:** `data/processed/lending_club_cleaned.csv`  
**Output:** Chart PNGs saved to `tableau/screenshots/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
CHART_DIR = "../tableau/screenshots"
os.makedirs(CHART_DIR, exist_ok=True)

df = pd.read_csv('../data/processed/lending_club_cleaned.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")    

## 3.1 — Default Rate Overview

In [ ]:
default_rate = df['loan_default'].mean() * 100
print(f"Overall Default Rate: {default_rate:.2f}%")
print(df['loan_default'].value_counts())
print(f"\n  Non-defaults: {(df['loan_default']==0).sum():,}")
print(f"  Defaults:     {(df['loan_default']==1).sum():,}")    

## 3.2 — Default Rate by Year (Time Series)

In [ ]:
yearly = df.groupby('issue_year')['loan_default'].mean() * 100
plt.figure(figsize=(12, 4))
plt.plot(yearly.index, yearly.values, marker='o', color='steelblue', linewidth=2)
plt.fill_between(yearly.index, yearly.values, alpha=0.1, color='steelblue')
plt.title('Default Rate by Loan Issue Year (2007–2020)', fontsize=14)
plt.xlabel('Year'); plt.ylabel('Default Rate (%)')
plt.xticks(yearly.index, rotation=45)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_default_by_year.png', dpi=150)
plt.show()
print(f"Peak default year: {yearly.idxmax()} ({yearly.max():.1f}%)")    

## 3.3 — Default Rate by Loan Grade

In [ ]:
grade_default = df.groupby('grade')['loan_default'].mean().sort_index() * 100
plt.figure(figsize=(8, 4))
sns.barplot(x=grade_default.index, y=grade_default.values,
            hue=grade_default.index, palette='RdYlGn_r', legend=False)
plt.title('Default Rate by Loan Grade', fontsize=14)
plt.xlabel('Grade'); plt.ylabel('Default Rate (%)')
for i, v in enumerate(grade_default.values):
    plt.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_default_by_grade.png', dpi=150)
plt.show()
print(grade_default)    

## 3.4 — Default Rate by US State (Top 20)

In [ ]:
state_default = (df.groupby('addr_state')['loan_default'].mean() * 100
                 .sort_values(ascending=False).head(20))
plt.figure(figsize=(12, 5))
sns.barplot(x=state_default.index, y=state_default.values,
            hue=state_default.index, palette='Reds_r', legend=False)
plt.title('Default Rate by US State (Top 20)', fontsize=14)
plt.xlabel('State'); plt.ylabel('Default Rate (%)')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_default_by_state.png', dpi=150)
plt.show()    

## 3.5 — Loan Amount Distribution

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df['loan_amnt'], bins=50, color='steelblue', kde=True)
plt.title('Loan Amount Distribution', fontsize=14)
plt.xlabel('Loan Amount ($)'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_loan_amnt_dist.png', dpi=150)
plt.show()
print(df['loan_amnt'].describe())    

## 3.6 — DTI vs Default

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x='loan_default', y='dti', data=df,
            hue='loan_default', palette={0: 'steelblue', 1: 'salmon'}, legend=False)
plt.title('Debt-to-Income Ratio vs Default', fontsize=14)
plt.xlabel('Loan Default (0=No, 1=Yes)'); plt.ylabel('DTI')
plt.xticks([0, 1], ['No Default', 'Default'])
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_dti_vs_default.png', dpi=150)
plt.show()

print("Mean DTI — No Default:", df[df['loan_default']==0]['dti'].mean().round(2))
print("Mean DTI — Default:   ", df[df['loan_default']==1]['dti'].mean().round(2))    

## 3.7 — Default Rate by Home Ownership

In [ ]:
own_default = (df.groupby('home_ownership')['loan_default'].mean() * 100
               .sort_values(ascending=False))
plt.figure(figsize=(8, 4))
sns.barplot(x=own_default.index, y=own_default.values,
            hue=own_default.index, palette='Blues_r', legend=False)
plt.title('Default Rate by Home Ownership', fontsize=14)
plt.xlabel('Ownership'); plt.ylabel('Default Rate (%)')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_default_by_ownership.png', dpi=150)
plt.show()    

## 3.8 — Default Rate by Loan Purpose

In [ ]:
purpose_default = (df.groupby('purpose')['loan_default'].mean() * 100
                   .sort_values(ascending=False))
plt.figure(figsize=(12, 5))
sns.barplot(y=purpose_default.index, x=purpose_default.values,
            hue=purpose_default.index, palette='coolwarm_r', legend=False)
plt.title('Default Rate by Loan Purpose', fontsize=14)
plt.xlabel('Default Rate (%)'); plt.ylabel('Purpose')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_default_by_purpose.png', dpi=150)
plt.show()    

## 3.9 — Correlation Heatmap

In [ ]:
num_cols = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'delinq_2yrs', 'revol_util', 'total_acc', 'loan_default',
    'emp_length_n', 'term_n', 'loan_to_income', 'credit_age_months'
]
available = [c for c in num_cols if c in df.columns]
corr = df[available].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Key Numeric Features', fontsize=14)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/eda_correlation_heatmap.png', dpi=150)
plt.show()

print("\nTop correlations with loan_default:")
print(corr['loan_default'].sort_values(ascending=False).drop('loan_default'))    

## Summary of EDA Findings

| Insight | Finding |
|---|---|
| Overall default rate | 13.08% |
| Peak default year | 2007 (financial crisis) |
| Highest default grade | G (≈45%) vs A (≈4%) |
| Interest rate correlation | Strongest numeric predictor (corr ≈ +0.21) |
| DTI | Defaulters have significantly higher DTI |
| Purpose | Small business loans default at highest rate |